# Example 2: Dask Service - Intro

The DEDL STACK Dask serivce was designed to enable data processing close to the data. Furthermore, a key requirement was the accessability of the service. An overview about the potential scenarios on how to access the service is given in the figure below.

<center>
<img src="./DEDL-STACK-Dask.png" width="75%">
</center>

## Dask Clusters
As a user of the DEDL STACK Dask service you will interact with remote **Dask Cluster** via **Dask Gateway**. Dask Gateway is a secure, multi-tenant server, enabling users to launch and manage Dask clusters. When we talk about Dask Clusters we can think of those as depicted in the following:

<center>
<img src="https://tutorial.dask.org/_images/distributed-overview.png" width="75%" alt="Distributed overview">
</center>


## How to spawn a Dask Cluster

Dask Gateway instances on the different DEDL bridges are accessible via dedicated HTTP API.
Dask Gateway provides a client library to interact with the various instances from with Python.

In addition, only authenticated access is granted to the DEDL STACK service Dask, therefore a helper class to authenticate a user against the DESP identity management system is implemented. The users password is directly handed over to the request object and is not permanently stored.

In the following, will authenticate with the DESP username and password against Dask Gateway hosted at LEONARDO bridge in Bologna.

>Again, the password will only be saved for the duration of this user session and will be remove as soon as the notebook/kernel is closed.


In [ ]:
from dask_gateway.auth import GatewayAuth
from getpass import getpass
from destinelab import AuthHandler as DESP_AuthHandler
from rich.prompt import Prompt
from dask_gateway import Gateway

class DESPAuth(GatewayAuth):
    def __init__(self, username: str):
        self.auth_handler = DESP_AuthHandler(username, getpass("Please input your DESP password: "))
        self.access_token = self.auth_handler.get_token()
    
    def pre_request(self, _):
        headers = {"Authorization": "Bearer " + self.access_token}
        return headers, None

myAuth = DESPAuth(username=Prompt.ask(prompt="Username"))
DaskLeonardo = Gateway(
    address="http://dask.leonardo.data.destination-earth.eu",
    proxy_address="tcp://dask.leonardo.data.destination-earth.eu:80",
    auth=myAuth)


We can now check if we already have a cluster running at LEONARDO bridge.

In [ ]:
dask_clusters_leonardo = DaskLeonardo.list_clusters()
dask_clusters_leonardo

DEDL STACK service Dask allows some custom configurations for the requested Dask Clusters. Resource limits are set by Dask Gateway depending on the assigned role to the user.

In [ ]:
options = DaskLeonardo.cluster_options()
options

Now we can spawn a cluster with the given configuration.

In [ ]:
cluster = DaskLeonardo.new_cluster(options)
cluster

Up to now the cluster will only consist of the distributed scheduler. If you want to spawn workers directly via Python adaptively, please use the following method call. With the following, the Dask cluster will be scaled to 2 workers initially. Depending on the load, Dask will add addtional workers, up to 5, if needed.

In [ ]:
cluster.adapt(minimum=2, maximum=5)

Now we ask for client to interact with the spawned cluster

In [ ]:
client = cluster.get_client()

A typical use case is a _Read-Transform-Write_ data workflow.
Most likely, one will implement this via a for-loop iterating over a list of files.

With Dask we can something similar, however, we can execute this in parallel.

**Now we will give it a try.**

In [ ]:
def process_file(filename):
    from time import sleep
    from random import randint

    # we simulate a processing
    sleep(randint(1, 10))
    
    return filename

We can run these function locally

In [ ]:
process_file('file1')

Or we can submit them to run remotely with Dask. This immediately returns a future that points to the ongoing computation, and eventually to the stored result.

In [ ]:
future = client.submit(process_file, 'New File processed remote')  # returns immediately with pending future
future

If you wait a second, and then check on the future again, you’ll see that it has finished.

In [ ]:
future

We want to get the actual result. You can block on the computation and gather the result with the .result() method.

In [ ]:
future.result()

We can also execute this over a list of files.

In [ ]:
n = 10  # dynamic length
file_list = [f"file{i}.geotiff" for i in range(1, n + 1)]
print(file_list)

We will make use of Dask Bag to map a function on all the files in the list.

In [ ]:
import dask.bag as db
from dask.diagnostics import ProgressBar

b = db.from_sequence(file_list)
mapped = b.map(process_file)

result = mapped.compute()
result

After we are done with our computations we can shutdonwn the cluster to free up resources.

In [ ]:
cluster.shutdown()